# HRM vs Baseline — Audio Language Model Head-to-Head

Trains two architectures with matched parameter count on LibriTTS-R first-codebook EnCodec tokens, and reports val-loss curves side-by-side.

Designed for **Kaggle T4×2** with the GPU runtime enabled. Total wall time ~6–10 hours for the default config.

**Setup before running:**
1. Settings → Accelerator → GPU T4×2
2. Settings → Internet → On
3. Upload this repo as a Kaggle Dataset, or clone via the cell below

In [ ]:
# 1. Pull the repo into /kaggle/working
import os, sys, subprocess, pathlib
ROOT = pathlib.Path('/kaggle/working/hrm-vall-e')
if not ROOT.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1',
                            'https://github.com/harrrshall/hrm-vall-e.git', str(ROOT)])
sys.path.insert(0, str(ROOT))
print('repo:', ROOT)

In [ ]:
# 2. Install dependencies (torch is pre-installed on Kaggle)
!pip install -q transformers==4.44.* accelerate==0.34.* datasets==2.21.* soundfile librosa torchaudio==2.4.*

In [ ]:
# 3. Tokenize a small LibriTTS-R subset with EnCodec (first codebook only)
# Default: ~1h of dev-clean. Bump --hours to 5.0 if you want a longer run.
import subprocess
DATA_DIR = pathlib.Path('/kaggle/working/data')
if not DATA_DIR.exists() or len(list(DATA_DIR.glob('sample_*.pt'))) == 0:
    subprocess.check_call([
        sys.executable, '-m', 'scripts.prepare_libritts',
        '--out', str(DATA_DIR),
        '--hours', '1.0',
        '--split', 'dev-clean',
    ], cwd=str(ROOT))
files = sorted(DATA_DIR.glob('sample_*.pt'))
print(f'{len(files)} clips tokenized')

In [ ]:
# 4. Train HRM and baseline back-to-back with matched params
import torch
from src.train import TrainConfig, build_model, lr_at, count_params
from src.data import LibriTTSEnCodecDataset, TokenLayout
from torch.utils.data import DataLoader, random_split

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, 'gpus:', torch.cuda.device_count())

# matched config (HRM h+l = baseline n_layers)
COMMON = dict(dim=384, num_heads=6, expansion=4.0, max_seq_len=512,
              batch_size=8, total_steps=5000, warmup_steps=200,
              lr=3e-4, log_every=50, device=device)
HRM_CFG = TrainConfig(backbone='hrm', n_h=4, n_l=4, h_cycles=2, l_cycles=3, **COMMON)
BASE_CFG = TrainConfig(backbone='baseline', n_layers=8, **COMMON)

layout = TokenLayout()
all_files = sorted(DATA_DIR.glob('sample_*.pt'))
split = int(0.9 * len(all_files))
train_files, val_files = all_files[:split], all_files[split:]
train_ds = LibriTTSEnCodecDataset(train_files, layout, max_seq_len=COMMON['max_seq_len'])
val_ds = LibriTTSEnCodecDataset(val_files, layout, max_seq_len=COMMON['max_seq_len'])
print(f'train: {len(train_ds)}  val: {len(val_ds)}')

In [ ]:
import math, time
from src.data import collate

def run_one(cfg: TrainConfig, train_ds, val_ds, name: str):
    torch.manual_seed(cfg.seed)
    model = build_model(cfg, layout).to(cfg.device)
    n_params = count_params(model)
    print(f'{name}: {n_params:,} params')
    tl = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate, drop_last=True, num_workers=2)
    vl = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate, drop_last=True, num_workers=2)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, betas=(cfg.beta1, cfg.beta2), weight_decay=cfg.weight_decay)
    hist = {'step': [], 'train_loss': [], 'val_loss': [], 'lr': [], 'bp_steps': []}
    model.train()
    step = 0; t0 = time.time()
    while step < cfg.total_steps:
        for batch in tl:
            if step >= cfg.total_steps: break
            batch = {k: v.to(cfg.device) for k, v in batch.items()}
            bp_steps = None
            if cfg.backbone == 'hrm':
                bp_steps = model.backbone.bp_steps_for(step, cfg.total_steps)
            lr = lr_at(step, cfg)
            for g in opt.param_groups: g['lr'] = lr
            out = model(batch['tokens'], prefix_mask=batch['prefix_mask'], labels=batch['labels'], bp_steps=bp_steps)
            opt.zero_grad(); out['loss'].backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            if step % cfg.log_every == 0:
                # quick val pass
                model.eval()
                with torch.no_grad():
                    vloss = sum(model(b['tokens'].to(cfg.device), prefix_mask=b['prefix_mask'].to(cfg.device), labels=b['labels'].to(cfg.device))['loss'].item() for b in vl) / len(vl)
                model.train()
                hist['step'].append(step); hist['train_loss'].append(out['loss'].item())
                hist['val_loss'].append(vloss); hist['lr'].append(lr); hist['bp_steps'].append(bp_steps)
                print(f"  step {step:>5d}  train {out['loss'].item():.3f}  val {vloss:.3f}  lr {lr:.2e}  bp {bp_steps}  {time.time()-t0:.0f}s")
            step += 1
    return {'model': model, 'hist': hist, 'n_params': n_params}

print('=== HRM ===');     hrm_run = run_one(HRM_CFG, train_ds, val_ds, 'HRM')
print('=== Baseline ==='); base_run = run_one(BASE_CFG, train_ds, val_ds, 'Baseline')

In [ ]:
# 5. Plot loss curves
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(hrm_run['hist']['step'], hrm_run['hist']['train_loss'], label='HRM')
ax[0].plot(base_run['hist']['step'], base_run['hist']['train_loss'], label='Baseline')
ax[0].set(title='train loss', xlabel='step', ylabel='loss'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(hrm_run['hist']['step'], hrm_run['hist']['val_loss'], label='HRM')
ax[1].plot(base_run['hist']['step'], base_run['hist']['val_loss'], label='Baseline')
ax[1].set(title='val loss', xlabel='step', ylabel='loss'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.suptitle(f"HRM ({hrm_run['n_params']:,} params)  vs  Baseline ({base_run['n_params']:,} params)")
plt.tight_layout()
plt.savefig('/kaggle/working/loss_curves.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"final val loss — HRM: {hrm_run['hist']['val_loss'][-1]:.4f}")
print(f"final val loss — Baseline: {base_run['hist']['val_loss'][-1]:.4f}")

In [ ]:
# 6. Save checkpoints
torch.save({'model': hrm_run['model'].state_dict(), 'hist': hrm_run['hist'], 'cfg': HRM_CFG.__dict__},
           '/kaggle/working/hrm_ckpt.pt')
torch.save({'model': base_run['model'].state_dict(), 'hist': base_run['hist'], 'cfg': BASE_CFG.__dict__},
           '/kaggle/working/baseline_ckpt.pt')
print('saved checkpoints to /kaggle/working/')